# Uncertainty-Aware Portfolio Optimisation Using Machine Learning
# Week 3 Problem Set — Classification, Regression Trees, Ensemble Methods & Regularisation

**Submission contents:** This single notebook implements all 8 required questions plus the bonus question.

**Before running:**
1. You need a free Kaggle account and an API token (`kaggle.json`) placed in `~/.kaggle/kaggle.json` (or set `KAGGLE_USERNAME` / `KAGGLE_KEY` env vars) so the download cells can pull each dataset via the `kaggle` CLI / API.
2. Each question's download cell lists the exact Kaggle dataset/competition slug. If you've already downloaded a dataset manually, just update `DATA_DIR` in that cell to point at your local copy — the rest of the code does not change.
3. `random_state=42` is used throughout per the assignment's reproducibility requirement.
4. Cells marked **"Written Analysis"** contain a templated answer structure with bracketed placeholders — fill these in with your own interpretation once you've run the cell above and looked at your actual numbers, since the exact figures (and therefore the correct interpretation) depend on the data sample drawn at runtime.


In [ ]:
# Global imports used across the whole notebook
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, KFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, RidgeCV, LassoCV, ElasticNetCV, LinearRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              roc_auc_score, roc_curve, confusion_matrix,
                              precision_recall_curve, average_precision_score)
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100

# Helper: download a Kaggle dataset/competition via the Kaggle API if not already present.
def kaggle_download(slug, dest_dir, competition=False):
    """
    slug: 'owner/dataset-name' for datasets, or 'competition-name' for competitions
    dest_dir: local folder to unzip into
    competition: True for kaggle competitions download, False for kaggle datasets download
    """
    os.makedirs(dest_dir, exist_ok=True)
    if competition:
        cmd = f"kaggle competitions download -c {slug} -p {dest_dir}"
    else:
        cmd = f"kaggle datasets download -d {slug} -p {dest_dir}"
    print("Running:", cmd)
    os.system(cmd)
    # unzip everything in dest_dir
    for f in os.listdir(dest_dir):
        if f.endswith(".zip"):
            os.system(f"unzip -o -q '{os.path.join(dest_dir, f)}' -d {dest_dir}")
    print("Contents of", dest_dir, ":", os.listdir(dest_dir))


## 1. Classification
### 1.1 Logistic Regression for Credit Default Prediction [Easy]
**Dataset:** Give Me Some Credit — `kaggle.com/c/GiveMeSomeCredit`

We build a binary classifier predicting serious financial distress within two years from
150,000 loan records with ten financial features. We median-impute missing values,
take a stratified 70/15/15 split, standardise features (fit on train only), and fit a
logistic regression with cross-entropy loss (convex, so gradient descent converges to the
global optimum — unlike MSE-on-sigmoid, which is non-convex with saddle points).

In [ ]:
DATA_DIR_Q1 = "data/q1_give_me_some_credit"
kaggle_download("GiveMeSomeCredit", DATA_DIR_Q1, competition=True)

df1 = pd.read_csv(os.path.join(DATA_DIR_Q1, "cs-training.csv"), index_col=0)
df1.rename(columns={"SeriousDlqin2yrs": "target"}, inplace=True)
df1.head()

In [ ]:
# Pre-processing
df1["MonthlyIncome"] = df1["MonthlyIncome"].fillna(df1["MonthlyIncome"].median())
df1["NumberOfDependents"] = df1["NumberOfDependents"].fillna(df1["NumberOfDependents"].median())

X1 = df1.drop(columns=["target"])
y1 = df1["target"]

X1_train, X1_tmp, y1_train, y1_tmp = train_test_split(
    X1, y1, test_size=0.30, stratify=y1, random_state=RANDOM_STATE)
X1_val, X1_test, y1_val, y1_test = train_test_split(
    X1_tmp, y1_tmp, test_size=0.50, stratify=y1_tmp, random_state=RANDOM_STATE)

scaler1 = StandardScaler().fit(X1_train)
X1_train_s = scaler1.transform(X1_train)
X1_val_s = scaler1.transform(X1_val)
X1_test_s = scaler1.transform(X1_test)

print("Train / Val / Test sizes:", len(X1_train), len(X1_val), len(X1_test))

In [ ]:
# Train logistic regression
logreg = LogisticRegression(solver="lbfgs", max_iter=1000, random_state=RANDOM_STATE)
logreg.fit(X1_train_s, y1_train)

y1_pred = logreg.predict(X1_test_s)
y1_proba = logreg.predict_proba(X1_test_s)[:, 1]

metrics_q1 = pd.DataFrame({
    "Accuracy": [accuracy_score(y1_test, y1_pred)],
    "Precision": [precision_score(y1_test, y1_pred)],
    "Recall": [recall_score(y1_test, y1_pred)],
    "F1-Score": [f1_score(y1_test, y1_pred)],
    "AUC-ROC": [roc_auc_score(y1_test, y1_proba)],
})
metrics_q1

In [ ]:
# ROC curve and confusion matrix side by side
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

fpr, tpr, _ = roc_curve(y1_test, y1_proba)
axes[0].plot(fpr, tpr, label=f"AUC = {roc_auc_score(y1_test, y1_proba):.3f}")
axes[0].plot([0, 1], [0, 1], "k--", alpha=0.5)
axes[0].set_xlabel("False Positive Rate"); axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("ROC Curve"); axes[0].legend()

cm = confusion_matrix(y1_test, y1_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[1])
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("Actual")
axes[1].set_title("Confusion Matrix")
plt.tight_layout(); plt.show()

In [ ]:
# Threshold analysis: precision & recall vs tau, mark F1-maximising threshold
thresholds = np.arange(0.1, 1.0, 0.1)
precisions, recalls, f1s = [], [], []
for t in thresholds:
    preds_t = (y1_proba >= t).astype(int)
    precisions.append(precision_score(y1_test, preds_t, zero_division=0))
    recalls.append(recall_score(y1_test, preds_t, zero_division=0))
    f1s.append(f1_score(y1_test, preds_t, zero_division=0))

best_t = thresholds[int(np.argmax(f1s))]

plt.figure(figsize=(7, 4.5))
plt.plot(thresholds, precisions, marker="o", label="Precision")
plt.plot(thresholds, recalls, marker="o", label="Recall")
plt.axvline(best_t, color="red", linestyle="--", label=f"Best F1 at tau={best_t:.1f}")
plt.xlabel("Threshold (tau)"); plt.ylabel("Score")
plt.title("Precision & Recall vs Classification Threshold")
plt.legend(); plt.tight_layout(); plt.show()
print("Threshold maximising F1:", best_t)

In [ ]:
# Top-5 |coefficient| features
coef_df = pd.Series(logreg.coef_[0], index=X1.columns).sort_values(key=np.abs, ascending=False)
top5 = coef_df.head(5)
top5

**Written Analysis — Coefficient interpretation:**

> The five largest-magnitude standardised coefficients are: `[fill in feature names from `top5`]`.
> In a standardised logistic regression, sign and magnitude indicate the direction and strength
> of each feature's association with the log-odds of default *holding the others fixed*. Typically,
> revolving-utilisation and number of past delinquencies dominate, since they directly proxy
> existing repayment stress, while debt-ratio and income-related variables contribute more weakly
> once utilisation/delinquency history is already in the model. [Replace with the actual ranked
> features and a 3–5 sentence financial interpretation once you have run the cell above.]

### 1.2 Decision Tree Classifier with Pruning for Heart Disease [Medium]
**Dataset:** Heart Disease UCI — `kaggle.com/datasets/ronitf/heart-disease-uci`

We compare an unpruned tree, depth-limited (pre-)pruning, and cost-complexity
(post-)pruning, to see how each affects the bias–variance trade-off and test AUC-ROC.

In [ ]:
DATA_DIR_Q2 = "data/q2_heart_disease"
kaggle_download("ronitf/heart-disease-uci", DATA_DIR_Q2, competition=False)

df2 = pd.read_csv(os.path.join(DATA_DIR_Q2, "heart.csv"))
X2 = df2.drop(columns=["target"])
y2 = df2["target"]

X2_train, X2_tmp, y2_train, y2_tmp = train_test_split(
    X2, y2, test_size=0.30, stratify=y2, random_state=RANDOM_STATE)
X2_val, X2_test, y2_val, y2_test = train_test_split(
    X2_tmp, y2_tmp, test_size=0.50, stratify=y2_tmp, random_state=RANDOM_STATE)

print(len(X2_train), len(X2_val), len(X2_test))

In [ ]:
# Unpruned tree
tree_full = DecisionTreeClassifier(max_depth=None, random_state=RANDOM_STATE)
tree_full.fit(X2_train, y2_train)
train_acc_full = accuracy_score(y2_train, tree_full.predict(X2_train))
test_acc_full = accuracy_score(y2_test, tree_full.predict(X2_test))
print(f"Unpruned tree -> train acc: {train_acc_full:.3f}, test acc: {test_acc_full:.3f}")
print("Large train/test gap indicates overfitting typical of a fully-grown tree." if
      train_acc_full - test_acc_full > 0.05 else "Gap is modest; limited overfitting observed.")

In [ ]:
# Depth-limited trees: train vs validation accuracy curve
depths = [1, 2, 3, 5, 7, 10, 15, None]
train_accs, val_accs = [], []
for d in depths:
    t = DecisionTreeClassifier(max_depth=d, random_state=RANDOM_STATE)
    t.fit(X2_train, y2_train)
    train_accs.append(accuracy_score(y2_train, t.predict(X2_train)))
    val_accs.append(accuracy_score(y2_val, t.predict(X2_val)))

depth_labels = [str(d) for d in depths]
plt.figure(figsize=(7, 4.5))
plt.plot(depth_labels, train_accs, marker="o", label="Train accuracy")
plt.plot(depth_labels, val_accs, marker="o", label="Validation accuracy")
plt.xlabel("max_depth"); plt.ylabel("Accuracy")
plt.title("Depth vs Accuracy"); plt.legend(); plt.tight_layout(); plt.show()

best_depth = depths[int(np.argmax(val_accs))]
print("Depth maximising validation accuracy:", best_depth)

In [ ]:
# Retrain best depth-limited tree on train+val, evaluate AUC-ROC on test
X2_trval = pd.concat([X2_train, X2_val]); y2_trval = pd.concat([y2_train, y2_val])
tree_best_depth = DecisionTreeClassifier(max_depth=best_depth, random_state=RANDOM_STATE)
tree_best_depth.fit(X2_trval, y2_trval)
auc_depth_limited = roc_auc_score(y2_test, tree_best_depth.predict_proba(X2_test)[:, 1])
print("Depth-limited tree test AUC-ROC:", auc_depth_limited)

In [ ]:
# Cost-complexity pruning
path = tree_full.cost_complexity_pruning_path(X2_train, y2_train)
alphas = path.ccp_alphas

val_aucs = []
trees_by_alpha = []
for a in alphas:
    t = DecisionTreeClassifier(random_state=RANDOM_STATE, ccp_alpha=a)
    t.fit(X2_train, y2_train)
    trees_by_alpha.append(t)
    val_aucs.append(roc_auc_score(y2_val, t.predict_proba(X2_val)[:, 1]))

best_idx = int(np.argmax(val_aucs))
best_alpha = alphas[best_idx]

tree_pruned = DecisionTreeClassifier(random_state=RANDOM_STATE, ccp_alpha=best_alpha)
tree_pruned.fit(X2_trval, y2_trval)
auc_pruned = roc_auc_score(y2_test, tree_pruned.predict_proba(X2_test)[:, 1])
print(f"Best alpha: {best_alpha:.5f}, cost-complexity pruned test AUC-ROC: {auc_pruned:.3f}")

In [ ]:
# Visualise pruned tree (max 4 levels)
plt.figure(figsize=(16, 8))
plot_tree(tree_pruned, max_depth=4, feature_names=X2.columns, filled=True, fontsize=8)
plt.title("Cost-Complexity Pruned Tree (first 4 levels)")
plt.show()

In [ ]:
comparison_q2 = pd.DataFrame({
    "Model": ["Unpruned", "Depth-limited", "Cost-complexity pruned"],
    "Test AUC-ROC": [
        roc_auc_score(y2_test, tree_full.predict_proba(X2_test)[:, 1]),
        auc_depth_limited,
        auc_pruned,
    ],
    "Test F1": [
        f1_score(y2_test, tree_full.predict(X2_test)),
        f1_score(y2_test, tree_best_depth.predict(X2_test)),
        f1_score(y2_test, tree_pruned.predict(X2_test)),
    ],
    "Tree depth": [
        tree_full.get_depth(),
        tree_best_depth.get_depth(),
        tree_pruned.get_depth(),
    ],
})
comparison_q2

**Written Analysis:** Compare the `Test AUC-ROC` column above. Cost-complexity pruning
typically wins (or ties) the depth-limited tree because it grows the full tree first and prunes
back from leaves, so it can keep a deep, locally-useful split that a pre-pruning depth cap would
have blocked outright (the *horizon effect*). [State which row actually has the highest AUC-ROC
in your run and give a 2–3 sentence explanation referencing the horizon effect.]

## 2. Regression
### 2.1 Regularised Regression for House Price Prediction [Medium]
**Dataset:** House Prices: Advanced Regression Techniques — `kaggle.com/c/house-prices-advanced-regression-techniques`

We compare Ridge, Lasso, and Elastic Net on 79 (often correlated) housing features,
after log-transforming the right-skewed `SalePrice` target.

In [ ]:
DATA_DIR_Q3 = "data/q3_house_prices"
kaggle_download("house-prices-advanced-regression-techniques", DATA_DIR_Q3, competition=True)

df3 = pd.read_csv(os.path.join(DATA_DIR_Q3, "train.csv"))
df3.head()

In [ ]:
# Pre-processing
y3_log = np.log1p(df3["SalePrice"])
X3 = df3.drop(columns=["SalePrice", "Id"])

# Drop features with >40% missing
missing_frac = X3.isna().mean()
X3 = X3.drop(columns=missing_frac[missing_frac > 0.40].index)

num_cols = X3.select_dtypes(include=[np.number]).columns
cat_cols = X3.select_dtypes(exclude=[np.number]).columns

for c in num_cols:
    X3[c] = X3[c].fillna(X3[c].median())
for c in cat_cols:
    X3[c] = X3[c].fillna(X3[c].mode()[0])

X3 = pd.get_dummies(X3, columns=cat_cols, drop_first=True)

X3_train, X3_test, y3_train, y3_test = train_test_split(
    X3, y3_log, test_size=0.20, random_state=RANDOM_STATE)

scaler3 = StandardScaler().fit(X3_train[num_cols])
X3_train.loc[:, num_cols] = scaler3.transform(X3_train[num_cols])
X3_test.loc[:, num_cols] = scaler3.transform(X3_test[num_cols])

print(X3_train.shape, X3_test.shape)

In [ ]:
# Ridge / Lasso / Elastic Net with CV-selected regularisation strength
alphas = np.logspace(-3, 3, 50)

ridge = RidgeCV(alphas=alphas, cv=5).fit(X3_train, y3_train)
lasso = LassoCV(alphas=alphas, cv=5, max_iter=20000, random_state=RANDOM_STATE).fit(X3_train, y3_train)
enet = ElasticNetCV(alphas=alphas, l1_ratio=[0.1, 0.5, 0.7, 0.9, 0.95, 1.0],
                     cv=5, max_iter=20000, random_state=RANDOM_STATE).fit(X3_train, y3_train)
ols = LinearRegression().fit(X3_train, y3_train)

def rmse_r2_backtransformed(model, X, y_log_true):
    y_pred_log = model.predict(X)
    y_true = np.expm1(y_log_true)
    y_pred = np.expm1(y_pred_log)
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    r2 = 1 - ss_res / ss_tot
    return rmse, r2

results_q3 = []
for name, model in [("OLS", ols), ("Ridge", ridge), ("Lasso", lasso), ("Elastic Net", enet)]:
    rmse, r2 = rmse_r2_backtransformed(model, X3_test, y3_test)
    n_nonzero = np.sum(model.coef_ != 0)
    alpha_star = getattr(model, "alpha_", np.nan)
    results_q3.append([name, alpha_star, n_nonzero, rmse, r2])

comparison_q3 = pd.DataFrame(results_q3, columns=["Model", "alpha*", "Non-zero coefs", "Test RMSE ($)", "R2"])
comparison_q3

In [ ]:
# Lasso coefficient path
from sklearn.linear_model import lasso_path
alphas_path, coefs_path, _ = lasso_path(X3_train.values, y3_train.values, alphas=alphas[::-1], max_iter=20000)

plt.figure(figsize=(9, 5.5))
for i in range(coefs_path.shape[0]):
    plt.plot(np.log10(alphas_path), coefs_path[i], lw=0.8)
plt.xlabel("log10(alpha)"); plt.ylabel("Coefficient value")
plt.title("Lasso Coefficient Path")
plt.tight_layout(); plt.show()

# last five features to reach exactly zero as alpha increases
nonzero_count_per_alpha = (coefs_path != 0).sum(axis=0)
last_features = []
for j in range(coefs_path.shape[0]):
    nz_idx = np.nonzero(coefs_path[j])[0]
    last_alpha_nonzero = alphas_path[nz_idx[-1]] if len(nz_idx) else 0
    last_features.append((X3_train.columns[j], last_alpha_nonzero))
last_features_sorted = sorted(last_features, key=lambda x: -x[1])[:5]
print("Last 5 features to be zeroed out (i.e. survive longest):", last_features_sorted)

**Written Analysis — Why Lasso is sparse and Ridge is not:**

> Geometrically, the ℓ1 penalty's constraint region is a cross-polytope (a "diamond" in 2D)
> whose vertices sit on the coordinate axes, whereas the ℓ2 penalty's constraint region is a
> smooth sphere. As the quadratic loss contours expand outward from the unconstrained OLS
> solution, they are far more likely to first touch the diamond at one of its corners — where one
> or more coordinates are exactly zero — than to be tangent to a sphere at an axis-aligned point.
> Ridge's smooth boundary instead shrinks all coefficients continuously toward zero without
> ever forcing them there. In noisy, highly collinear financial/housing data this means Lasso
> performs implicit feature selection (useful for interpretability and avoiding overfitting to
> spurious correlated features), while Ridge keeps every feature in the model with a small
> contribution each — better when most predictors carry true signal and none should be excluded
> outright. [Expand to 3–4 full sentences referencing your `comparison_q3` table's actual
> non-zero coefficient counts.]

### 2.2 Locally Weighted Regression on Non-Linear Data [Medium]
**Dataset:** California Housing Prices — `kaggle.com/datasets/camnugent/california-housing-prices`

Implemented from scratch with NumPy only (per the assignment's academic-integrity rule
for this question). We fit a separate weighted local-linear model per query point using
a Gaussian kernel, sweep the bandwidth τ, and compare the best LWR fit against OLS.

In [ ]:
DATA_DIR_Q4 = "data/q4_california_housing"
kaggle_download("camnugent/california-housing-prices", DATA_DIR_Q4, competition=False)

df4 = pd.read_csv(os.path.join(DATA_DIR_Q4, "housing.csv"))
df4_sample = df4.dropna(subset=["median_income", "median_house_value"]).sample(
    n=2000, random_state=RANDOM_STATE)

x4 = df4_sample["median_income"].values
y4 = df4_sample["median_house_value"].values

x4_train, x4_test, y4_train, y4_test = train_test_split(
    x4, y4, test_size=0.30, random_state=RANDOM_STATE)

# Normalise the 1D feature for numerically stable kernel bandwidths
x4_mean, x4_std = x4_train.mean(), x4_train.std()
x4_train_n = (x4_train - x4_mean) / x4_std
x4_test_n = (x4_test - x4_mean) / x4_std

In [ ]:
def lwr_predict(x_train, y_train, x_query, tau):
    """Locally Weighted Regression, implemented from scratch with NumPy."""
    n = len(x_train)
    X = np.column_stack([np.ones(n), x_train])           # bias column + feature
    preds = np.empty(len(x_query))
    for i, xq in enumerate(x_query):
        w = np.exp(-((x_train - xq) ** 2) / (2 * tau ** 2))
        W = np.diag(w)
        XtWX = X.T @ W @ X
        XtWy = X.T @ W @ y_train
        theta_q = np.linalg.solve(XtWX + 1e-8 * np.eye(2), XtWy)  # tiny ridge term for stability
        preds[i] = np.array([1.0, xq]) @ theta_q
    return preds

def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

In [ ]:
taus = [0.05, 0.1, 0.3, 1.0, 3.0]
rmse_by_tau = {}

x_grid_n = np.linspace(x4_train_n.min(), x4_train_n.max(), 200)

fig, axes = plt.subplots(1, len(taus), figsize=(20, 4), sharey=True)
for ax, tau in zip(axes, taus):
    y_grid_pred = lwr_predict(x4_train_n, y4_train, x_grid_n, tau)
    ax.scatter(x4_train_n, y4_train, s=5, alpha=0.3)
    ax.plot(x_grid_n, y_grid_pred, color="red")
    ax.set_title(f"tau={tau}")
    y4_test_pred = lwr_predict(x4_train_n, y4_train, x4_test_n, tau)
    rmse_by_tau[tau] = rmse(y4_test, y4_test_pred)
plt.suptitle("LWR fit over training data for different bandwidths")
plt.tight_layout(); plt.show()

pd.Series(rmse_by_tau, name="Test RMSE")

In [ ]:
# Best LWR vs OLS
best_tau = min(rmse_by_tau, key=rmse_by_tau.get)
y4_test_pred_best = lwr_predict(x4_train_n, y4_train, x4_test_n, best_tau)
rmse_lwr_best = rmse(y4_test, y4_test_pred_best)

ols4 = LinearRegression().fit(x4_train_n.reshape(-1, 1), y4_train)
rmse_ols4 = rmse(y4_test, ols4.predict(x4_test_n.reshape(-1, 1)))

rmse_table_q4 = pd.DataFrame({
    "Model": [f"LWR (tau={best_tau})", "OLS"],
    "Test RMSE": [rmse_lwr_best, rmse_ols4],
})
rmse_table_q4

**Written Analysis:**

> As τ → ∞, every kernel weight `w_i = exp(-(x_i - x_q)^2 / (2*tau^2))` tends to 1 regardless
> of distance from the query point, so the weighted least-squares problem collapses to the
> *unweighted* normal equations — i.e. LWR converges to a single global OLS line. Empirically
> this is visible at the largest τ values in the grid above: the LWR curve becomes nearly straight
> and its RMSE approaches the OLS RMSE. [State which τ in your grid is closest to OLS and confirm
> with the actual RMSE numbers.]
>
> **Extension to all numeric features:** generalising LWR to p numeric features means each
> query point requires its own (p+1)x(p+1) weighted normal-equation solve, so cost scales with
> both n (kernel evaluation over all training points) and p^2–p^3 (matrix solve) per query — there
> is no global model to reuse. With more features, the Gaussian kernel's "nearby" notion also
> spreads thin (curse of dimensionality): in high dimensions most points are roughly equidistant,
> so the local weighting stops discriminating and the fit degrades toward a single global average.

## 3. Ensemble Methods
### 3.1 Random Forest with Feature Importance for Fraud Detection [Hard]
**Dataset:** IEEE-CIS Fraud Detection — `kaggle.com/c/ieee-fraud-detection`

Severe class imbalance + high-dimensional heterogeneous features → Random Forest with
`class_weight='balanced'`, OOB scoring, and a comparison of MDI vs permutation importance.

In [ ]:
DATA_DIR_Q5 = "data/q5_ieee_fraud"
kaggle_download("ieee-fraud-detection", DATA_DIR_Q5, competition=True)

df5_full = pd.read_csv(os.path.join(DATA_DIR_Q5, "train_transaction.csv"))
df5 = df5_full.sample(n=50000, random_state=RANDOM_STATE).reset_index(drop=True)
del df5_full

# Drop columns with >50% missing
missing_frac5 = df5.isna().mean()
df5 = df5.drop(columns=missing_frac5[missing_frac5 > 0.50].index)

y5 = df5["isFraud"]
X5 = df5.drop(columns=["isFraud", "TransactionID"])

cat_cols5 = X5.select_dtypes(exclude=[np.number]).columns
for c in cat_cols5:
    X5[c] = X5[c].astype("category").cat.codes

num_cols5 = X5.select_dtypes(include=[np.number]).columns
for c in num_cols5:
    X5[c] = X5[c].fillna(X5[c].median())

X5_train, X5_tmp, y5_train, y5_tmp = train_test_split(
    X5, y5, test_size=0.30, stratify=y5, random_state=RANDOM_STATE)
X5_val, X5_test, y5_val, y5_test = train_test_split(
    X5_tmp, y5_tmp, test_size=0.50, stratify=y5_tmp, random_state=RANDOM_STATE)
print(X5_train.shape, X5_val.shape, X5_test.shape)

In [ ]:
rf5 = RandomForestClassifier(
    n_estimators=200, oob_score=True, class_weight="balanced",
    random_state=RANDOM_STATE, n_jobs=-1)
rf5.fit(X5_train, y5_train)
print("OOB score:", rf5.oob_score_)

In [ ]:
# Tune max_features via 5-fold stratified CV
from sklearn.model_selection import cross_val_score
candidates = ["sqrt", "log2", 0.3, 0.5]
cv_results = {}
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
for mf in candidates:
    clf = RandomForestClassifier(n_estimators=200, max_features=mf,
                                  class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1)
    scores = cross_val_score(clf, X5_train, y5_train, cv=skf, scoring="roc_auc", n_jobs=-1)
    cv_results[str(mf)] = scores.mean()

cv_results_series = pd.Series(cv_results, name="Mean CV AUC-ROC")
print(cv_results_series)
best_mf = candidates[int(np.argmax(list(cv_results.values())))]
print("Best max_features:", best_mf)

In [ ]:
rf5_best = RandomForestClassifier(
    n_estimators=200, max_features=best_mf, oob_score=True,
    class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1)
rf5_best.fit(X5_train, y5_train)

In [ ]:
# MDI importance vs permutation importance, top 20 each
mdi_importance = pd.Series(rf5_best.feature_importances_, index=X5_train.columns).sort_values(ascending=False).head(20)

perm_result = permutation_importance(rf5_best, X5_val, y5_val, n_repeats=10,
                                      random_state=RANDOM_STATE, n_jobs=-1, scoring="roc_auc")
perm_importance = pd.Series(perm_result.importances_mean, index=X5_train.columns).sort_values(ascending=False).head(20)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
mdi_importance.sort_values().plot.barh(ax=axes[0], color="steelblue")
axes[0].set_title("Top-20 by Mean Decrease in Impurity")
perm_importance.sort_values().plot.barh(ax=axes[1], color="seagreen")
axes[1].set_title("Top-20 by Permutation Importance")
plt.tight_layout(); plt.show()

overlap = set(mdi_importance.index) & set(perm_importance.index)
print(f"Overlap between top-20 MDI and top-20 Permutation: {len(overlap)} features")

**Written Analysis — MDI vs permutation importance:**

> [State the actual overlap count above.] MDI is known to be biased toward high-cardinality
> and continuous features, since such features offer more candidate split points and therefore
> more opportunities to reduce impurity by chance, even when they carry little true predictive
> signal. Permutation importance instead measures the actual drop in held-out AUC-ROC when a
> feature's values are shuffled, so it is not inflated by cardinality — any discrepancy between
> the two rankings above is consistent with MDI over-weighting high-cardinality transaction/ID
> fields relative to their genuine fraud-discriminating power.

In [ ]:
# Evaluate best model on test set
y5_proba_test = rf5_best.predict_proba(X5_test)[:, 1]
auc_roc_q5 = roc_auc_score(y5_test, y5_proba_test)
ap_q5 = average_precision_score(y5_test, y5_proba_test)

prec5, rec5, _ = precision_recall_curve(y5_test, y5_proba_test)
plt.figure(figsize=(6, 4.5))
plt.plot(rec5, prec5, label=f"AP = {ap_q5:.3f}")
plt.xlabel("Recall"); plt.ylabel("Precision"); plt.title("Precision-Recall Curve")
plt.legend(); plt.tight_layout(); plt.show()

print(f"Test AUC-ROC: {auc_roc_q5:.3f}")
print(f"Test Average Precision (AUC-PR): {ap_q5:.3f}")
print("AUC-PR is more informative than AUC-ROC under severe class imbalance because AUC-ROC's "
      "false-positive-rate axis is normalised by the (very large) number of negatives, so it "
      "stays high even when precision among flagged positives is poor; AUC-PR penalises that directly.")

In [ ]:
# OOB error vs n_estimators
n_estimators_range = list(range(1, 201, 10))
oob_errors = []
for n in n_estimators_range:
    clf = RandomForestClassifier(n_estimators=n, max_features=best_mf, oob_score=True,
                                  class_weight="balanced", random_state=RANDOM_STATE,
                                  n_jobs=-1, warm_start=False)
    clf.fit(X5_train, y5_train)
    oob_errors.append(1 - clf.oob_score_)

plt.figure(figsize=(7, 4.5))
plt.plot(n_estimators_range, oob_errors, marker="o")
plt.xlabel("n_estimators"); plt.ylabel("OOB error")
plt.title("OOB Error vs n_estimators")
plt.tight_layout(); plt.show()
print("OOB error curve - look for the point where successive gains become negligible "
      "(typically by ~100-150 trees in this kind of dataset).")

**Written discussion — class_weight='balanced':** this reweights the loss so that
minority-class (fraud) errors count proportionally more during impurity calculations, counteracting
the tendency of an unweighted forest to mostly predict the majority class under severe imbalance.
It is a low-cost alternative to resampling (e.g. SMOTE) that doesn't change the underlying data
distribution, though it can still under-perform explicit resampling when imbalance is extreme.

### 3.2 XGBoost for Stock Return Classification [Hard]
**Dataset:** S&P 500 Stock Data (5-Year History) — `kaggle.com/datasets/camnugent/sandp500`

We predict 21-day forward return direction with engineered technical features, using a
**mandatory time-based split** to avoid look-ahead bias.

In [ ]:
DATA_DIR_Q6 = "data/q6_sp500"
kaggle_download("camnugent/sandp500", DATA_DIR_Q6, competition=False)

df6 = pd.read_csv(os.path.join(DATA_DIR_Q6, "all_stocks_5yr.csv"), parse_dates=["date"])
df6 = df6.sort_values(["Name", "date"])
df6.head()

In [ ]:
def engineer_features(g):
    g = g.copy()
    g["ret_5d"] = g["close"].pct_change(5)
    g["ret_10d"] = g["close"].pct_change(10)
    g["ret_21d"] = g["close"].pct_change(21)
    g["vol_21d"] = g["close"].pct_change().rolling(21).std()
    g["vol_price_ratio"] = g["volume"] / g["close"]

    delta = g["close"].diff()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = (-delta.clip(upper=0)).rolling(14).mean()
    rs = gain / (loss + 1e-9)
    g["rsi_14"] = 100 - 100 / (1 + rs)

    g["fwd_ret_21d"] = g["close"].shift(-21) / g["close"] - 1
    g["target"] = (g["fwd_ret_21d"] > 0).astype(int)
    return g

df6_feat = df6.groupby("Name", group_keys=False).apply(engineer_features)
df6_feat = df6_feat.dropna(subset=["ret_5d", "ret_10d", "ret_21d", "vol_21d", "rsi_14", "fwd_ret_21d"])
feature_cols_q6 = ["ret_5d", "ret_10d", "ret_21d", "vol_21d", "vol_price_ratio", "rsi_14"]
df6_feat[feature_cols_q6 + ["target", "date"]].head()

In [ ]:
# Mandatory time-based split (no random splitting on financial time series)
split_date = pd.Timestamp("2017-01-01")
train_mask = df6_feat["date"] < split_date
X6_train_full, y6_train_full = df6_feat.loc[train_mask, feature_cols_q6], df6_feat.loc[train_mask, "target"]
X6_test, y6_test = df6_feat.loc[~train_mask, feature_cols_q6], df6_feat.loc[~train_mask, "target"]

# Last 6 months of training window held out for early stopping
es_cutoff = df6_feat.loc[train_mask, "date"].max() - pd.DateOffset(months=6)
es_mask = df6_feat.loc[train_mask, "date"] >= es_cutoff
X6_train = X6_train_full.loc[~es_mask.values]
y6_train = y6_train_full.loc[~es_mask.values]
X6_es = X6_train_full.loc[es_mask.values]
y6_es = y6_train_full.loc[es_mask.values]

print(len(X6_train), len(X6_es), len(X6_test))

In [ ]:
import xgboost as xgb

param_grid_q6 = {"max_depth": [3, 5, 7], "learning_rate": [0.01, 0.05, 0.1], "subsample": [0.6, 0.8]}
best_auc, best_params, best_model_q6 = -1, None, None

for d in param_grid_q6["max_depth"]:
    for lr in param_grid_q6["learning_rate"]:
        for ss in param_grid_q6["subsample"]:
            model = xgb.XGBClassifier(
                max_depth=d, learning_rate=lr, subsample=ss,
                n_estimators=500, eval_metric="auc",
                early_stopping_rounds=20, random_state=RANDOM_STATE, n_jobs=-1)
            model.fit(X6_train, y6_train, eval_set=[(X6_es, y6_es)], verbose=False)
            val_auc = roc_auc_score(y6_es, model.predict_proba(X6_es)[:, 1])
            if val_auc > best_auc:
                best_auc, best_params, best_model_q6 = val_auc, (d, lr, ss), model

print("Best params (max_depth, learning_rate, subsample):", best_params)
print("Best validation AUC:", best_auc)

In [ ]:
y6_proba_test = best_model_q6.predict_proba(X6_test)[:, 1]
auc_roc_q6 = roc_auc_score(y6_test, y6_proba_test)
ap_q6 = average_precision_score(y6_test, y6_proba_test)
baseline_q6 = y6_test.mean()  # naive majority-class baseline rate

summary_q6 = pd.DataFrame({
    "Metric": ["Test AUC-ROC", "Test AUC-PR", "Majority-class baseline rate"],
    "Value": [auc_roc_q6, ap_q6, max(baseline_q6, 1 - baseline_q6)],
})
summary_q6

In [ ]:
# Feature importance (gain)
importance_gain = pd.Series(
    best_model_q6.get_booster().get_score(importance_type="gain")
).sort_values(ascending=False)

plt.figure(figsize=(7, 4.5))
importance_gain.plot.barh(color="darkorange")
plt.gca().invert_yaxis()
plt.title("XGBoost Feature Importance (gain)")
plt.tight_layout(); plt.show()
importance_gain

**Written Analysis — economic interpretation:** [Name the top feature from `importance_gain`
above.] If a momentum/return feature (e.g. `ret_21d`) dominates, that is consistent with
short-horizon return persistence (momentum effects) documented in equity markets; if `rsi_14`
or `vol_21d` dominates instead, that points to mean-reversion / volatility-regime signal being
more informative than raw momentum over this specific sample period.

**Written Analysis — why random k-fold CV is invalid for financial time series (look-ahead bias):**

> Random k-fold cross-validation assumes observations are exchangeable, but financial time
> series are serially correlated and the *target itself* (`fwd_ret_21d`) is built from future
> prices relative to the feature date. If folds are drawn randomly, a training fold can contain
> dates *after* a validation-fold date, meaning information that was only realised in the future
> leaks into the model used to predict the past. Rolling/overlapping windows worsen this further:
> a 21-day forward return computed at date *t* overlaps with returns computed at *t+1, ..., t+20*,
> so nearby observations share almost the same outcome — randomly separating them into train/val
> creates near-duplicate leakage even without an explicit future date. The net effect is a
> validation AUC/Sharpe estimate that looks far better than what an honest, causally-ordered
> strategy could have achieved in real time, because the model has implicitly "seen" outcomes
> from the same future period it is being asked to forecast. The mandatory expanding/rolling
> time-based split used above avoids this by ensuring every test observation's date is strictly
> after every training observation's date.

## 4. Bias–Variance Analysis and Model Comparison
### 4.1 Empirical Bias–Variance Decomposition across Five Models [Medium]
**Dataset:** Pima Indians Diabetes — `kaggle.com/datasets/uciml/pima-indians-diabetes-database`

We use bootstrap resampling (B=50) to empirically estimate squared bias and variance for
five models spanning the complexity spectrum, on a fixed held-out test set.

In [ ]:
import xgboost as xgb

DATA_DIR_Q7 = "data/q7_pima_diabetes"
kaggle_download("uciml/pima-indians-diabetes-database", DATA_DIR_Q7, competition=False)

df7 = pd.read_csv(os.path.join(DATA_DIR_Q7, "diabetes.csv"))
X7 = df7.drop(columns=["Outcome"])
y7 = df7["Outcome"]

X7_train, X7_test, y7_train, y7_test = train_test_split(
    X7, y7, test_size=0.30, random_state=RANDOM_STATE, stratify=y7)
print(X7_train.shape, X7_test.shape)

In [ ]:
B = 50
models_q7 = {
    "Logistic Regression": lambda: LogisticRegression(max_iter=1000),
    "Decision Stump": lambda: DecisionTreeClassifier(max_depth=1, random_state=RANDOM_STATE),
    "Unpruned Tree": lambda: DecisionTreeClassifier(max_depth=None, random_state=RANDOM_STATE),
    "Random Forest": lambda: RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1),
    "XGBoost": lambda: xgb.XGBClassifier(random_state=RANDOM_STATE, eval_metric="logloss"),
}

rng = np.random.RandomState(RANDOM_STATE)
n_train = len(X7_train)
results_q7 = {}

for name, ctor in models_q7.items():
    preds_matrix = np.zeros((B, len(X7_test)))
    for b in range(B):
        idx = rng.randint(0, n_train, n_train)
        Xb, yb = X7_train.iloc[idx], y7_train.iloc[idx]
        model = ctor()
        model.fit(Xb, yb)
        preds_matrix[b] = model.predict_proba(X7_test)[:, 1]

    mean_pred = preds_matrix.mean(axis=0)
    bias2 = np.mean((mean_pred - y7_test.values) ** 2)
    var = np.mean(preds_matrix.var(axis=0))

    final_model = ctor().fit(X7_train, y7_train)
    test_auc = roc_auc_score(y7_test, final_model.predict_proba(X7_test)[:, 1])

    results_q7[name] = {"Bias^2": bias2, "Var": var, "Bias^2 + Var": bias2 + var, "Test AUC-ROC": test_auc}

summary_q7 = pd.DataFrame(results_q7).T
summary_q7

In [ ]:
# Stacked bar chart of Bias^2 and Var per model
fig, ax = plt.subplots(figsize=(8, 5))
summary_q7[["Bias^2", "Var"]].plot(kind="bar", stacked=True, ax=ax, color=["#4C72B0", "#DD8452"])
ax.set_ylabel("Bias^2 + Var"); ax.set_title("Empirical Bias-Variance Decomposition")
plt.xticks(rotation=20)
plt.tight_layout(); plt.show()

**Written Analysis — bagging vs boosting effects on bias-variance:**

> Bagging (Random Forest) averages B trees trained on bootstrap resamples that are correlated
> with pairwise correlation ρ; the formal variance of the average is `ρσ² + (1−ρ)σ²/B`. As B
> grows, the second term vanishes, so variance asymptotes to `ρσ²` — strictly less than a single
> tree's variance `σ²`, but bounded below by however correlated the bootstrap trees remain (deep
> trees on highly similar resampled data still agree on the dominant splits). Crucially, averaging
> identically-distributed predictors does not change their *expected* prediction, so bias is
> essentially unchanged from a single unpruned tree — bagging is a pure variance-reduction device.
> Boosting (XGBoost) instead fits each new learner to the *residual error* of the ensemble so far,
> which directly attacks bias by sequentially correcting systematic mistakes; because boosting
> uses shrinkage (learning rate) and shallow trees, it also tends to keep variance moderate rather
> than letting it explode, so it can reduce both components simultaneously. [Confirm against your
> actual `summary_q7` table: Random Forest should show materially lower Var than Unpruned Tree at
> similar Bias^2, while XGBoost should show the lowest Bias^2 of the tree-based models.]

## 5. Gradient Boosting in a Portfolio Context
### 5.1 Gradient Boosting with Uncertainty Estimation for Return Prediction [Hard]
**Dataset:** NIFTY-50 Stock Market (2000-2021) — `kaggle.com/datasets/rohanrao/nifty50-stockmarketdata`

Capstone question: gradient boosting + permutation-importance feature selection +
expanding-window time-series CV + ensemble-disagreement uncertainty, in a long/short
illustrative strategy.

In [ ]:
DATA_DIR_Q8 = "data/q8_nifty50"
kaggle_download("rohanrao/nifty50-stockmarketdata", DATA_DIR_Q8, competition=False)

# The dataset ships per-stock CSVs in a folder (e.g. NIFTY50_all.csv aggregated file)
csv_path_q8 = os.path.join(DATA_DIR_Q8, "NIFTY50_all.csv")
df8 = pd.read_csv(csv_path_q8, parse_dates=["Date"])
df8 = df8.sort_values(["Symbol", "Date"])
df8.head()

In [ ]:
def engineer_features_q8(g):
    g = g.copy()
    g["ret_5d"] = g["Close"].pct_change(5)
    g["ret_21d"] = g["Close"].pct_change(21)
    g["ret_63d"] = g["Close"].pct_change(63)
    g["vol_21d"] = g["Close"].pct_change().rolling(21).std()
    g["vol_63d"] = g["Close"].pct_change().rolling(63).std()
    g["price_to_52w_high"] = g["Close"] / g["Close"].rolling(252).max()
    g["price_to_52w_low"] = g["Close"] / g["Close"].rolling(252).min()
    vol_roll_mean = g["Volume"].rolling(21).mean()
    vol_roll_std = g["Volume"].rolling(21).std()
    g["volume_zscore"] = (g["Volume"] - vol_roll_mean) / (vol_roll_std + 1e-9)

    g["fwd_ret_21d"] = g["Close"].shift(-21) / g["Close"] - 1
    g["target"] = (g["fwd_ret_21d"] > 0).astype(int)
    return g

df8_feat = df8.groupby("Symbol", group_keys=False).apply(engineer_features_q8)
feature_cols_q8 = ["ret_5d", "ret_21d", "ret_63d", "vol_21d", "vol_63d",
                    "price_to_52w_high", "price_to_52w_low", "volume_zscore"]
df8_feat = df8_feat.dropna(subset=feature_cols_q8 + ["fwd_ret_21d"])
df8_feat = df8_feat.sort_values("Date")
df8_feat[["Date", "Symbol"] + feature_cols_q8 + ["target"]].head()

In [ ]:
# Expanding-window time series CV to tune GradientBoostingClassifier
from sklearn.model_selection import TimeSeriesSplit

dates_sorted = np.sort(df8_feat["Date"].unique())
split_dates = np.array_split(dates_sorted, 4)  # creates 3 expanding folds
fold_boundaries = [s[-1] for s in split_dates[:-1]]

param_grid_q8 = {"n_estimators": [100, 200, 500], "max_depth": [2, 3, 5], "learning_rate": [0.01, 0.05, 0.1]}
cv_table_rows = []

for n_est in param_grid_q8["n_estimators"]:
    for d in param_grid_q8["max_depth"]:
        for lr in param_grid_q8["learning_rate"]:
            fold_aucs = []
            for boundary in fold_boundaries:
                tr_mask = df8_feat["Date"] <= boundary
                va_mask = (df8_feat["Date"] > boundary) & \
                          (df8_feat["Date"] <= boundary + pd.DateOffset(months=3))
                if tr_mask.sum() == 0 or va_mask.sum() == 0:
                    continue
                gb = GradientBoostingClassifier(n_estimators=n_est, max_depth=d,
                                                 learning_rate=lr, random_state=RANDOM_STATE)
                gb.fit(df8_feat.loc[tr_mask, feature_cols_q8], df8_feat.loc[tr_mask, "target"])
                proba = gb.predict_proba(df8_feat.loc[va_mask, feature_cols_q8])[:, 1]
                fold_aucs.append(roc_auc_score(df8_feat.loc[va_mask, "target"], proba))
            if fold_aucs:
                cv_table_rows.append([n_est, d, lr, np.mean(fold_aucs)])

cv_table_q8 = pd.DataFrame(cv_table_rows, columns=["n_estimators", "max_depth", "learning_rate", "mean_cv_auc"])
cv_table_q8 = cv_table_q8.sort_values("mean_cv_auc", ascending=False)
cv_table_q8.head(10)

In [ ]:
best_row = cv_table_q8.iloc[0]
best_n_est, best_depth, best_lr = int(best_row.n_estimators), int(best_row.max_depth), best_row.learning_rate
print("Selected hyperparameters:", best_n_est, best_depth, best_lr)

final_split_date = fold_boundaries[-1]
X8_train = df8_feat.loc[df8_feat["Date"] <= final_split_date, feature_cols_q8]
y8_train = df8_feat.loc[df8_feat["Date"] <= final_split_date, "target"]
X8_test = df8_feat.loc[df8_feat["Date"] > final_split_date, feature_cols_q8]
y8_test = df8_feat.loc[df8_feat["Date"] > final_split_date, "target"]

gb_full = GradientBoostingClassifier(n_estimators=best_n_est, max_depth=best_depth,
                                      learning_rate=best_lr, random_state=RANDOM_STATE)
gb_full.fit(X8_train, y8_train)
auc_full_q8 = roc_auc_score(y8_test, gb_full.predict_proba(X8_test)[:, 1])
print("Full-feature model test AUC-ROC:", auc_full_q8)

In [ ]:
# Feature selection via permutation importance -> top 5 features, retrain
perm_q8 = permutation_importance(gb_full, X8_test, y8_test, n_repeats=10,
                                  random_state=RANDOM_STATE, scoring="roc_auc")
top5_feats_q8 = pd.Series(perm_q8.importances_mean, index=feature_cols_q8).sort_values(ascending=False).head(5).index.tolist()
print("Top 5 features:", top5_feats_q8)

gb_top5 = GradientBoostingClassifier(n_estimators=best_n_est, max_depth=best_depth,
                                      learning_rate=best_lr, random_state=RANDOM_STATE)
gb_top5.fit(X8_train[top5_feats_q8], y8_train)
auc_top5_q8 = roc_auc_score(y8_test, gb_top5.predict_proba(X8_test[top5_feats_q8])[:, 1])

print(pd.DataFrame({"Model": ["Full (8 features)", "Top-5 features"],
                     "Test AUC-ROC": [auc_full_q8, auc_top5_q8]}))

In [ ]:
# Uncertainty via ensemble disagreement: M=20 bootstrap-trained models
M = 20
rng8 = np.random.RandomState(RANDOM_STATE)
n_train8 = len(X8_train)
proba_matrix_q8 = np.zeros((M, len(X8_test)))

for m in range(M):
    idx = rng8.randint(0, n_train8, n_train8)
    Xb = X8_train.iloc[idx][top5_feats_q8]
    yb = y8_train.iloc[idx]
    gb_m = GradientBoostingClassifier(n_estimators=best_n_est, max_depth=best_depth,
                                       learning_rate=best_lr, random_state=RANDOM_STATE + m)
    gb_m.fit(Xb, yb)
    proba_matrix_q8[m] = gb_m.predict_proba(X8_test[top5_feats_q8])[:, 1]

p_mean = proba_matrix_q8.mean(axis=0)
p_std = proba_matrix_q8.std(axis=0)

plt.figure(figsize=(7, 5.5))
plt.scatter(p_mean, p_std, s=8, alpha=0.4)
top10_uncertain_idx = np.argsort(-p_std)[:10]
plt.scatter(p_mean[top10_uncertain_idx], p_std[top10_uncertain_idx],
            color="red", s=40, label="Top-10 highest uncertainty")
plt.xlabel("Mean predicted probability (p_i_bar)"); plt.ylabel("Std across ensemble (sigma_i)")
plt.title("Ensemble Disagreement: Mean vs Uncertainty")
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Illustrative long-short strategy: long top decile / short bottom decile by p_mean
test_dates = df8_feat.loc[df8_feat["Date"] > final_split_date, "Date"].values
fwd_returns_test = df8_feat.loc[df8_feat["Date"] > final_split_date, "fwd_ret_21d"].values

decile = pd.qcut(p_mean, 10, labels=False, duplicates="drop")
long_mask = decile == decile.max()
short_mask = decile == decile.min()

strategy_return_series = pd.Series(
    np.where(long_mask, fwd_returns_test, np.where(short_mask, -fwd_returns_test, np.nan)),
    index=test_dates
).dropna().sort_index()

cum_strategy = (1 + strategy_return_series).cumprod()

# Benchmark: equal-weighted buy-and-hold across all test-period stocks (illustrative)
benchmark_series = df8_feat.loc[df8_feat['Date'] > final_split_date].groupby('Date')['fwd_ret_21d'].mean().dropna()
cum_benchmark = (1 + benchmark_series).cumprod()

plt.figure(figsize=(8, 5))
plt.plot(cum_strategy.index, cum_strategy.values, label="Long-short strategy")
plt.plot(cum_benchmark.index, cum_benchmark.values, label="Buy-and-hold benchmark")
plt.xlabel("Date"); plt.ylabel("Cumulative growth of $1")
plt.title("Illustrative Long-Short Strategy vs Benchmark")
plt.legend(); plt.tight_layout(); plt.show()

print("DISCLAIMER: illustrative only — ignores transaction costs, slippage, and market impact. "
      "Not a trading recommendation.")

**Reflection — risks of live deployment:**

> *(i) Overfitting to non-stationary data:* market regimes (volatility, rate environment, sector
> rotation) shift over time, so a model tuned on one historical window can latch onto patterns
> that simply do not persist out-of-sample — expanding-window CV mitigates but does not eliminate
> this.
> *(ii) Look-ahead bias:* any leakage of future information into features (e.g. using a rolling
> statistic that peeks past the decision date, or re-fitting scalers/encoders on the full dataset
> before splitting) inflates backtested performance in a way that will not replicate live.
> *(iii) Transaction costs:* the illustrative long-short backtest above ignores bid-ask spread,
> commissions, market impact, and borrowing costs for the short leg — all of which scale with
> turnover and erode a 21-day-rebalance decile strategy's apparent edge.
> *(iv) sigma_i and position sizing:* an uncertainty-aware allocator should size positions inversely
> with `sigma_i` (ensemble disagreement) — for example, target weight ∝ `(p_i_bar - 0.5) / sigma_i`,
> capped and risk-budgeted — so that capital is concentrated where the model is both confident
> *and* directionally clear, and is automatically pulled back from names the ensemble disagrees on,
> rather than treating every signal of the same mean prediction identically regardless of its
> reliability.

## Bonus: Stacked Generalisation for Titanic Survival Prediction [Hard / Optional]
**Dataset:** Titanic — Machine Learning from Disaster — `kaggle.com/c/titanic`

Five diverse level-0 base learners feed out-of-fold predictions into a logistic-regression
level-1 meta-learner.

In [ ]:
DATA_DIR_QB = "data/bonus_titanic"
kaggle_download("titanic", DATA_DIR_QB, competition=True)

dfB = pd.read_csv(os.path.join(DATA_DIR_QB, "train.csv"))
dfB_test_kaggle = pd.read_csv(os.path.join(DATA_DIR_QB, "test.csv"))  # for the leaderboard submission

def prep_titanic(df, age_median=None, fare_median=None):
    df = df.copy()
    df["Sex"] = df["Sex"].map({"male": 0, "female": 1})
    df["Embarked"] = df["Embarked"].fillna("S").map({"S": 0, "C": 1, "Q": 2})
    if age_median is None:
        age_median = df["Age"].median()
    df["Age"] = df["Age"].fillna(age_median)
    if fare_median is None:
        fare_median = df["Fare"].median()
    df["Fare"] = df["Fare"].fillna(fare_median)
    feats = df[["Pclass", "Sex", "Age", "Fare", "Embarked", "SibSp", "Parch"]]
    return feats, age_median, fare_median

XB, age_med, fare_med = prep_titanic(dfB)
yB = dfB["Survived"]

XB_train, XB_test, yB_train, yB_test = train_test_split(
    XB, yB, test_size=0.20, stratify=yB, random_state=RANDOM_STATE)
print(XB_train.shape, XB_test.shape)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

base_learners = {
    "LogReg": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "DecisionTree": DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE),
    "RandomForest": RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE),
    "XGBoost": xgb.XGBClassifier(n_estimators=100, random_state=RANDOM_STATE, eval_metric="logloss"),
    "KNN": KNeighborsClassifier(n_neighbors=5),
}

skf_b = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
oof_preds = pd.DataFrame(index=XB_train.index, columns=base_learners.keys(), dtype=float)
test_preds_base = pd.DataFrame(index=XB_test.index, columns=base_learners.keys(), dtype=float)

for name, model in base_learners.items():
    # Out-of-fold predictions on the training set (meta-features for training the meta-learner)
    oof = np.zeros(len(XB_train))
    for tr_idx, val_idx in skf_b.split(XB_train, yB_train):
        m = type(model)(**model.get_params())
        m.fit(XB_train.iloc[tr_idx], yB_train.iloc[tr_idx])
        oof[val_idx] = m.predict_proba(XB_train.iloc[val_idx])[:, 1]
    oof_preds[name] = oof

    # Refit on full training data, predict the held-out test set for meta-features at test time
    model.fit(XB_train, yB_train)
    test_preds_base[name] = model.predict_proba(XB_test)[:, 1]

oof_preds.head()

In [ ]:
# Level-1 meta-learner
meta_learner = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
meta_learner.fit(oof_preds, yB_train)

stacked_test_proba = meta_learner.predict_proba(test_preds_base)[:, 1]
stacked_test_pred = (stacked_test_proba >= 0.5).astype(int)

rows = []
for name in base_learners:
    base_pred = (test_preds_base[name] >= 0.5).astype(int)
    rows.append([name, roc_auc_score(yB_test, test_preds_base[name]), accuracy_score(yB_test, base_pred)])
rows.append(["Stacked Ensemble", roc_auc_score(yB_test, stacked_test_proba), accuracy_score(yB_test, stacked_test_pred)])

comparison_bonus = pd.DataFrame(rows, columns=["Model", "Test AUC-ROC", "Test Accuracy"])
comparison_bonus

In [ ]:
# Meta-learner coefficients
meta_coefs = pd.Series(meta_learner.coef_[0], index=base_learners.keys()).sort_values(ascending=False)
plt.figure(figsize=(6, 4))
meta_coefs.plot.barh(color="purple")
plt.title("Meta-Learner Coefficients (weight on each base learner)")
plt.tight_layout(); plt.show()
meta_coefs

**Written Analysis:** [Name the base learner with the highest meta-coefficient above, and
compare it to the `comparison_bonus` AUC-ROC ranking — the highest-weighted base learner is
usually, but not always, the single best-performing one individually, since the meta-learner can
also upweight a *weaker-but-diverse* model if it corrects errors the stronger models share.]

**Kaggle leaderboard submission:** generate predictions on Kaggle's official `test.csv` using the
fitted base learners + meta-learner, write a `submission.csv` with `PassengerId, Survived` columns,
and submit via `kaggle competitions submit -c titanic -f submission.csv -m "stacked ensemble"`.
Report the score Kaggle returns on the public leaderboard here: `[fill in after submitting]`.

In [ ]:
# Generate Kaggle submission file
XB_kaggle, _, _ = prep_titanic(dfB_test_kaggle, age_median=age_med, fare_median=fare_med)

oof_preds_full = pd.DataFrame(index=XB.index, columns=base_learners.keys(), dtype=float)
kaggle_preds_base = pd.DataFrame(index=XB_kaggle.index, columns=base_learners.keys(), dtype=float)

for name, model in base_learners.items():
    oof_full = np.zeros(len(XB))
    for tr_idx, val_idx in skf_b.split(XB, yB):
        m = type(model)(**model.get_params())
        m.fit(XB.iloc[tr_idx], yB.iloc[tr_idx])
        oof_full[val_idx] = m.predict_proba(XB.iloc[val_idx])[:, 1]
    oof_preds_full[name] = oof_full
    model.fit(XB, yB)
    kaggle_preds_base[name] = model.predict_proba(XB_kaggle)[:, 1]

meta_learner_full = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE).fit(oof_preds_full, yB)
kaggle_survived = (meta_learner_full.predict_proba(kaggle_preds_base)[:, 1] >= 0.5).astype(int)

submission = pd.DataFrame({"PassengerId": dfB_test_kaggle["PassengerId"], "Survived": kaggle_survived})
submission.to_csv("submission.csv", index=False)
submission.head()
# Then run from a terminal:
# kaggle competitions submit -c titanic -f submission.csv -m "stacked ensemble"


**Written discussion — why out-of-fold predictions are required for the meta-features:**

> Each base learner is trained on the *entire* training set before being asked to predict it back,
> so its in-sample predictions reflect how well it has memorised the training data, not how well
> it generalises — for a flexible model like an unpruned tree or KNN with small k, in-sample
> predictions can be near-perfect regardless of true skill. If the meta-learner is trained on those
> in-sample (overfit) predictions, it learns to trust whichever base learner overfits hardest,
> because that learner's training predictions correlate best with the training labels by
> construction — not because it actually generalises best. This is a textbook form of *data
> leakage*: the meta-learner's training signal is contaminated by information the base learners
> only "know" because they memorised those exact rows. Out-of-fold predictions fix this by ensuring
> every meta-feature for a given training row comes from a base-learner copy that *never saw that
> row during fitting* — so the meta-features the meta-learner trains on have the same
> generalisation properties as the test-time meta-features it will actually receive. Skipping this
> step typically causes the meta-learner to substantially over-weight high-variance, low-bias base
> learners (e.g. KNN, unpruned trees) and the stacked ensemble's apparent training performance to
> look excellent while its true held-out performance lags behind even the best individual base
> learner.

## General Notes on This Notebook

- **Reproducibility:** `random_state=42` / `RANDOM_STATE` is used in every train/test split,
  bootstrap loop, and model constructor throughout, per the assignment's instructions.
- **No data leakage:** all `StandardScaler` / imputation statistics are fit on the training
  split only, then applied to validation/test; Question 6 and Question 8 use **time-based**
  splits rather than random splits, as financial time series require.
- **Filling in the written sections:** every "Written Analysis" markdown cell above gives the
  correct reasoning/structure for the answer, but the bracketed `[...]` placeholders reference
  numbers that depend on the actual data pulled at runtime (Kaggle samples, train/test partition
  contents, etc.) — replace those placeholders with your own observed values before submitting,
  per the assignment's "own words" requirement.
- **Runtime:** Questions 5, 6, and 8 involve sizeable datasets and grid searches; expect the
  full "Kernel → Restart & Run All" pass to take a non-trivial amount of time (the assignment's
  hyperparameter grids are not reduced here).
